In [ ]:
import sys
import os
import torch
import glob

from datasets import load_dataset
from torch_geometric.datasets import QM9
from torch_geometric.data import Data

from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import rdmolfiles

import numpy as np
import pandas as pd

In [ ]:
from rdkit.Chem import rdDetermineBonds


def safe_qm9_xyz_to_pyg(xyz_file_path):
    try:
        with open(xyz_file_path, 'r') as f:
            lines = f.readlines()

        # QM9-Format-Validierung
        num_atoms = int(lines[0].strip())

        # Zeile 2 enthält die QM9-Eigenschaften (Labels) als Text
        properties_line = lines[1].strip().split()

        # Beispiel: HOMO/LUMO extrahieren (falls Sie diese benötigen)
        # In QM9 liegen HOMO/LUMO oft an Index 7 und 8 der Eigenschaften-Zeile
        homo = float(properties_line[7]) if len(properties_line) > 7 else 0.0
        lumo = float(properties_line[8]) if len(properties_line) > 8 else 0.0
        y = torch.tensor([[homo, lumo]], dtype=torch.float) # Zielwert-Tensor

        # Wir bauen einen sauberen XYZ-Block NUR mit den Atomen für RDKit
        # Standard XYZ benötigt: Zeile 1 = Atomanzahl, Zeile 2 = Kommentar, danach Atome
        clean_xyz_block = f"{num_atoms}\nCleaned QM9 Conformer\n"

        # Nur die Atomzeilen auslesen (startet bei Zeile 3 bis num_atoms + 2)
        for i in range(2, 2 + num_atoms):
            # QM9-XYZ verwendet oft Tabulatoren, wir vereinheitlichen das auf Leerzeichen
            parts = lines[i].strip().split()
            # Element, X, Y, Z extrahieren (QM9 hat am Ende der Zeile oft noch Partialladungen, die ignorieren wir)
            atom_symbol = parts[0]
            x_coord, y_coord, z_coord = parts[1], parts[2], parts[3]
            clean_xyz_block += f"{atom_symbol} {x_coord} {y_coord} {z_coord}\n"

        # 1. RDKit-Molekül aus sauberem String-Block erzeugen
        mol = Chem.MolFromXYZBlock(clean_xyz_block)
        if mol is None:
            return None

        # 2. Bindungen präzise bestimmen (rdDetermineBonds ist robuster als DetermineConnectivity)
        rdDetermineBonds.DetermineBonds(mol, charge=0)

        # 3. Validieren
        Chem.SanitizeMol(mol)

        # 4. In PyTorch Geometric konvertieren
        atoms = [atom.GetAtomicNum() for atom in mol.GetAtoms()]
        x = torch.tensor(atoms, dtype=torch.long).view(-1, 1)

        conformer = mol.GetConformer()
        pos = torch.tensor(conformer.GetPositions(), dtype=torch.float)

        edge_indices = []
        for bond in mol.GetBonds():
            i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
            edge_indices.append([i, j])
            edge_indices.append([j, i])
        edge_index = torch.tensor(edge_indices, dtype=torch.long).t().contiguous()

        # Rückgabe des fertigen Graphen inklusive der Zielwerte (y)
        return Data(x=x, pos=pos, edge_index=edge_index, y=y)

    except Exception as e:
        # Fängt verbleibende, echte Formatierungsfehler ab
        print(f"Fehler bei Datei {os.path.basename(xyz_file_path)}: {e}")
        return None


In [ ]:

# Pfad zu Ihrem Ordner mit den XYZ-Dateien
xyz_folder = r"C:\Users\chris\PycharmProjects\ML_project\data\qm9\qm9_xyz"

# Alle .xyz-Dateien im Ordner finden
xyz_files = sorted(glob.glob(os.path.join(xyz_folder, "*.xyz")))

print(f"{len(xyz_files)} XYZ-Dateien gefunden. Starte Schleife...")

pyg_dataset = []

# Jede Datei einzeln an Ihre Methode übergeben
for path in xyz_files:
    data = safe_qm9_xyz_to_pyg(path)

    if data is not None:
        pyg_dataset.append(data)

print(f"Fertig! {len(pyg_dataset)} Moleküle erfolgreich verarbeitet.")

In [ ]:
pyg2 = pyg_dataset.copy()


In [ ]:
len(pyg2)

In [ ]:
import os
import glob
import torch
from rdkit import Chem
from rdkit.Chem import rdDetermineBonds
from torch_geometric.data import Data
from rdkit import RDLogger

# 1. HIER WERDEN DIE STÖRENDEN FEHLER STUMMGESCHALTET:
RDLogger.DisableLog('rdApp.*')

def safe_qm9_xyz_to_pyg(xyz_file_path, file_name):
    try:
        with open(xyz_file_path, 'r') as f:
            lines = f.readlines()

        num_atoms = int(lines[0].strip())
        properties_line = lines[1].strip().split()

        homo = float(properties_line[7]) if len(properties_line) > 7 else 0.0
        lumo = float(properties_line[8]) if len(properties_line) > 8 else 0.0
        y = torch.tensor([[homo, lumo]], dtype=torch.float)

        clean_xyz_block = f"{num_atoms}\nCleaned QM9 Conformer\n"

        for i in range(2, 2 + num_atoms):
            parts = lines[i].strip().split()
            atom_symbol = parts[0]
            x_coord, y_coord, z_coord = parts[1], parts[2], parts[3]
            clean_xyz_block += f"{atom_symbol} {x_coord} {y_coord} {z_coord}\n"

        mol = Chem.MolFromXYZBlock(clean_xyz_block)
        if mol is None:
            return None

        rdDetermineBonds.DetermineBonds(mol, charge=0)
        Chem.SanitizeMol(mol)

        atoms = [atom.GetAtomicNum() for atom in mol.GetAtoms()]
        x = torch.tensor(atoms, dtype=torch.long).view(-1, 1)

        conformer = mol.GetConformer()
        pos = torch.tensor(conformer.GetPositions(), dtype=torch.float)

        edge_indices = []
        for bond in mol.GetBonds():
            i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
            edge_indices.append([i, j])
            edge_indices.append([j, i])
        edge_index = torch.tensor(edge_indices, dtype=torch.long).t().contiguous()

        # Nutzt jetzt den bereinigten gdb_X Namen
        return Data(x=x, pos=pos, edge_index=edge_index, y=y, name=file_name)

    except Exception:
        return None

if __name__ == "__main__":
    # Pfade anpassen
    xyz_folder = r"C:\Users\chris\PycharmProjects\ML_project\data\qm9\qm9_xyz"
    speicher_pfad = r"C:\Users\chris\PycharmProjects\ML_project\data\qm9\qm9_graphen.pt"

    xyz_files = sorted(glob.glob(os.path.join(xyz_folder, "*.xyz")))

    print(f"{len(xyz_files)} XYZ-Dateien gefunden. Starte Konvertierung...")
    print("Bitte warten, das Terminal bleibt jetzt ruhig, während die Daten verarbeitet werden...")

    pyg_dataset = []

    for idx, path in enumerate(xyz_files):
        # 1. Reinen Dateinamen holen (z.B. "dsgdb9nsd_000001")
        raw_name = os.path.splitext(os.path.basename(path))[0]

        # 2. Umwandlung in das "gdb_X" Format
        try:
            # Holt die Zahl am Ende, entfernt führende Nullen: "000001" -> 1
            mol_num = int(raw_name.split('_')[-1])
            gdb_name = f"gdb_{mol_num}"
        except Exception:
            gdb_name = raw_name # Fallback, falls das Splitten fehlschlägt

        data = safe_qm9_xyz_to_pyg(path, gdb_name)
        if data is not None:
            pyg_dataset.append(data)

        if (idx + 1) % 20000 == 0:
            print(f"Fortschritt: {idx + 1} von {len(xyz_files)} Dateien geschafft...")

    print(f"\nGeschafft! {len(pyg_dataset)} von {len(xyz_files)} Molekülen erfolgreich konvertiert.")

    torch.save(pyg_dataset, speicher_pfad)
    print(f"Datensatz erfolgreich gespeichert unter: {speicher_pfad}")


In [ ]:
import torch
from torch_geometric.loader import DataLoader

# 1. Den fertig konvertierten Datensatz blitzschnell laden
dataset_path = r"/data/QM9\qm9_graphen.pt"
dataset = torch.load(dataset_path, weights_only=False)

print(f"Datensatz erfolgreich geladen! Anzahl Graphen: {len(dataset)}")

# Ein einzelnes Molekül zur Kontrolle anschauen
beispiel_graph = dataset[1]
print("\nBeispiel-Molekül Struktur:")
print(f"Anzahl Atome (Knoten): {beispiel_graph.num_nodes}")
print(f"Atom-Features (x): {beispiel_graph.x.shape}")
print(f"Zielwerte (y - HOMO/LUMO): {beispiel_graph.y}")

# 2. DataLoader für das spätere GNN-Training erstellen
# Batch-Size steuert, wie viele Moleküle gleichzeitig verarbeitet werden
#loader = DataLoader(dataset, batch_size=32, shuffle=True)


In [ ]:
import pandas as pd
import torch

# 1. Pfade definieren
csv_path = r"/data/QM9\gdb9.sdf.csv"
pt_dataset_path = r"/data/QM9\qm9_graphen.pt"
output_csv_path = r"/data/QM9\qm9_filtered2.csv"

# 2. CSV-Datei und .pt-Datensatz laden
print("Lade CSV und PT-Datei...")
df = pd.read_csv(csv_path)
dataset = torch.load(pt_dataset_path, weights_only=False)

# WICHTIG: Setzen Sie hier den exakten Namen der ID-Spalte aus Ihrer CSV ein!
column_with_names = 'mol_id'

# 3. Alle Namen aus der PT-Datei und der CSV extrahieren
survived_names = set(mol.name for mol in dataset if hasattr(mol, 'name'))
all_csv_names = set(df[column_with_names].astype(str))

print(f"Ursprüngliche Moleküle in CSV: {len(df)}")
print(f"Überlebende Moleküle in PT-Datei: {len(survived_names)}")

# --- NEU: Berechnung der entfernten Dateien ---
# Wir ziehen die überlebenden IDs von allen CSV-IDs ab
removed_names = all_csv_names - survived_names
# Sortieren für eine schönere Anzeige im Terminal
removed_names_sorted = sorted(list(removed_names))

print(f"Anzahl entfernter Moleküle: {len(removed_names_sorted)}")
# ----------------------------------------------

# 4. Die CSV filtern
df_filtered = df[df[column_with_names].astype(str).isin(survived_names)]
print(f"Gefilterte Moleküle in neuer CSV: {len(df_filtered)}")

# 5. Die perfekt angeglichene CSV-Datei speichern
df_filtered.to_csv(output_csv_path, index=False)
print(f"Erfolgreich angeglichene CSV gespeichert unter: {output_csv_path}\n")

# --- NEU: Ausgabe der entfernten GDB-Nummern ---
if removed_names_sorted:
    print("=== FOLGENDE GDB-NUMMERN WURDEN ENTFERNT ===")
    # Wenn es sehr viele sind, zeigen wir die ersten 100 und die letzten an,
    # damit das Terminal nicht komplett überflutet wird
    if len(removed_names_sorted) > 100:
        for name in removed_names_sorted[:50]:
            print(f"  {name}")
        print("  ... [weitere Einträge ausgeblendet] ...")
        for name in removed_names_sorted[-50:]:
            print(f"  {name}")
    else:
        for name in removed_names_sorted:
            print(f"  {name}")
    print("============================================")
else:
    print("Info: Es wurden keine Moleküle entfernt. Beide Datensätze waren bereits identisch.")


In [ ]:
dataset_path2 = r"C:\Users\chris\PycharmProjects\ML_project\tests\data\QM9\raw\qm9_v3.pt"
dataset2 = torch.load(dataset_path2, weights_only=False)

In [ ]:
dataset2[0]

In [ ]:

# sys.modules['rdkit'] = None

dataset = QM9(root='data/QM9')

print(f"Erfolgreich geladen! Anzahl Moleküle: {len(dataset)}")


In [ ]:
ds = load_dataset("HR-machine/QM9-Dataset")

In [ ]:
# Have a look at the molecules in the sdf file
supplier = Chem.SDMolSupplier(r"/data/QM9\gdb9.sdf")

count = 0

for mol in supplier:
    if mol is None:
        continue

    print("SMILES:", Chem.MolToSmiles(mol))
    print("Anzahl Atome:", mol.GetNumAtoms())
    print("Anzahl Bindungen:", mol.GetNumBonds())

    # 🔬 Atom-spezifische Infos
    for atom in mol.GetAtoms():
        print(
            atom.GetIdx(),
            atom.GetSymbol(),
            "Degree:", atom.GetDegree(),
            "Charge:", atom.GetFormalCharge(),
            "Hybrid:", atom.GetHybridization()
        )

    print("-----")

    count += 1
    if count >= 3:
        break

In [ ]:
# 1. Konvertiere jeden Split in ein Pandas DataFrame
df_train = ds["train"].to_pandas()
df_validation = ds["validation"].to_pandas()
df_test = ds["test"].to_pandas()

# 2. Verbinde alle DataFrames zu einem einzigen großen DataFrame
df_all = pd.concat([df_train, df_validation, df_test], ignore_index=True)

print(f"Gesamtanzahl der Moleküle: {len(df_all)}")

In [ ]:
# 1. Zahl aus der mol_id extrahieren (regulärer Ausdruck \d+ sucht nach Ziffern)
df_all['sort_helper'] = df_all['mol_id'].str.extract(r'(\d+)').astype(int)

# 2. Nach der Zahl sortieren und den Index aufräumen
df_all = df_all.sort_values(by='sort_helper').reset_index(drop=True)

# 3. Die temporäre Hilfsspalte wieder löschen
df_all = df_all.drop(columns=['sort_helper'])


In [ ]:
def create_morgan(smiles):

    mol = Chem.MolFromSmiles(smiles)

    fingerprint = AllChem.GetMorganFingerprintAsBitVect(
        mol,
        radius=2,
        nBits=2048
    )

    array = np.zeros((2048,))

    Chem.DataStructs.ConvertToNumpyArray(
        fingerprint,
        array
    )

    return array

In [ ]:
# 1. Das erste Molekül holen
sample = dataset[0]

# 2. Den SMILES-Code extrahieren
smiles_code = sample.smiles
print(f"SMILES für das erste Molekül: {smiles_code}") # Gibt 'C' (für Methan) aus

# 3. In Ihre Funktion füttern
fingerprint = create_morgan(smiles_code)
print(f"Fingerprint-Form: {fingerprint.shape}") # Gibt (2048,) aus


In [ ]:
ethanol = "CCO"

fp = create_morgan(ethanol)

print(fp.shape)